In [1]:
import sys
sys.path.append("/home/mike/git/QDMpy/src")

In [2]:
from QDMpy import models

12:32:46.596     INFO QDMpy.<module> >> WELCOME TO QDMpy
12:32:46.597    DEBUG QDMpy.<module> >> QDMpy version 0.1.0a installed at /home/mike/git/QDMpy/src/QDMpy
12:32:46.597    DEBUG QDMpy.<module> >> QDMpy config file /home/mike/.config/QDMpy/config.ini
12:32:46.598     INFO QDMpy.load_config >> Loading config file: /home/mike/.config/QDMpy/config.ini
0 <class 'ctypes.c_ulong'>
1 <class 'ctypes.c_ulong'>
2 <class 'pygpufit.gpufit.LP_c_float'>
3 <class 'pygpufit.gpufit.LP_c_float'>
4 <class 'ctypes.c_int'>
5 <class 'pygpufit.gpufit.LP_c_float'>
6 <class 'pygpufit.gpufit.LP_c_float'>
7 <class 'pygpufit.gpufit.LP_c_int'>
8 <class 'ctypes.c_float'>
9 <class 'ctypes.c_int'>
10 <class 'pygpufit.gpufit.LP_c_int'>
11 <class 'ctypes.c_int'>
12 <class 'ctypes.c_ulong'>
13 <class 'ctypes.LP_c_char'>
14 <class 'pygpufit.gpufit.LP_c_float'>
15 <class 'pygpufit.gpufit.LP_c_int'>
16 <class 'pygpufit.gpufit.LP_c_float'>
17 <class 'pygpufit.gpufit.LP_c_int'>
12:32:46.599     INFO QDMpy.<module> >> CU

# ODMR Spectral Models Tutorial

This tutorial demonstrates how to understand and use the ODMR spectral models in QDMpy for fitting nitrogen-vacancy (NV) center data.

## Overview of ODMR Models

QDMpy provides three built-in spectral models for fitting ODMR data from nitrogen-vacancy centers:

1. **ESR14N** - For NV centers with ¹⁴N isotope (3 dips)
2. **ESR15N** - For NV centers with ¹⁵N isotope (2 dips)  
3. **ESRSINGLE** - For single resonance systems (1 dip)

These models are implemented using Lorentzian lineshapes and are optimized for GPU-accelerated fitting.

---

## Model Architecture

The model system in QDMpy consists of:
- **Model Functions**: Core mathematical functions (`esr14n`, `esr15n`, `esrsingle`)
- **Model Classes**: Object-oriented wrappers (`ESR14N`, `ESR15N`, `ESRSINGLE`)
- **ModelRegistry**: Central registry for model management and retrieval

---

## Understanding the Available Models

Let's explore each model and understand their physics and parameters.

## Exploring Model Properties

Let's examine each model's characteristics:

In [ ]:
from QDMpy.models import ModelRegistry

# List all available models
all_models = ModelRegistry.all()
print("Available models:")
for name, info in all_models.items():
    print(f"  {name}: {info['class'].__name__}")

print("\nDetailed model information:")
for model_name in ['ESR14N', 'ESR15N', 'ESRSINGLE']:
    model = ModelRegistry.get(model_name)
    print(f"\n{model_name}:")
    print(f"  Parameters: {model.n_parameters}")
    print(f"  Peaks: {model.n_peaks}")
    print(f"  Parameter names: {model.parameters_unique}")
    print(f"  Hyperfine constant: {all_models[model_name]['hyp']} Hz")

## Visualizing Model Responses

Let's create realistic ODMR spectra using each model:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Create realistic frequency array (around 2.87 GHz)
frequencies = np.linspace(2.86e9, 2.88e9, 1000)

# Create subplots for comparison
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

# ESRSINGLE Model
model_single = ModelRegistry.get('ESRSINGLE')
# Parameters: [center, width, contrast, offset]
params_single = np.array([2.870e9, 3e6, 0.15, 0.0])
spectrum_single = model_single.func(frequencies, params_single)

axes[0].plot(frequencies/1e9, spectrum_single[0], 'b-', linewidth=2)
axes[0].set_title('ESRSINGLE Model - Single Lorentzian Dip')
axes[0].set_ylabel('Normalized Fluorescence')
axes[0].grid(True, alpha=0.3)
axes[0].text(2.8605, 0.5, f'Parameters: {model_single.n_parameters}\nPeaks: {model_single.n_peaks}', 
             bbox=dict(boxstyle="round,pad=0.3", facecolor="lightblue"))

# ESR15N Model  
model_15n = ModelRegistry.get('ESR15N')
# Parameters: [center, width, contrast_-1/2, contrast_+1/2, offset]
params_15n = np.array([2.870e9, 3e6, 0.12, 0.12, 0.0])
spectrum_15n = model_15n.func(frequencies, params_15n)

axes[1].plot(frequencies/1e9, spectrum_15n[0], 'g-', linewidth=2)
axes[1].set_title('ESR15N Model - ¹⁵N Isotope (I=1/2) Doublet')
axes[1].set_ylabel('Normalized Fluorescence')
axes[1].grid(True, alpha=0.3)
axes[1].text(2.8605, 0.5, f'Parameters: {model_15n.n_parameters}\nPeaks: {model_15n.n_peaks}\nHyperfine: {all_models["ESR15N"]["hyp"]} Hz', 
             bbox=dict(boxstyle="round,pad=0.3", facecolor="lightgreen"))

# ESR14N Model
model_14n = ModelRegistry.get('ESR14N')
# Parameters: [center, width, contrast_-1, contrast_0, contrast_+1, offset]  
params_14n = np.array([2.870e9, 3e6, 0.08, 0.15, 0.08, 0.0])
spectrum_14n = model_14n.func(frequencies, params_14n)

axes[2].plot(frequencies/1e9, spectrum_14n[0], 'r-', linewidth=2)
axes[2].set_title('ESR14N Model - ¹⁴N Isotope (I=1) Triplet')
axes[2].set_xlabel('Frequency (GHz)')
axes[2].set_ylabel('Normalized Fluorescence')
axes[2].grid(True, alpha=0.3)
axes[2].text(2.8605, 0.5, f'Parameters: {model_14n.n_parameters}\nPeaks: {model_14n.n_peaks}\nHyperfine: {all_models["ESR14N"]["hyp"]} Hz', 
             bbox=dict(boxstyle="round,pad=0.3", facecolor="lightcoral"))

plt.tight_layout()
plt.show()

## Model Selection Guidelines

### When to use each model:

**ESR14N**:
- Working with natural diamond (99% ¹⁴N isotope)
- Well-resolved hyperfine structure visible
- Need to fit all three hyperfine components independently

**ESR15N**:
- Working with isotopically enriched ¹⁵N diamond
- Two-peak structure is clearly visible
- Smaller hyperfine splitting than ¹⁴N

**ESRSINGLE**:
- Highly broadened spectra (e.g., due to strain or high temperature)
- Proof-of-concept measurements
- When hyperfine structure is not resolved
- Fitting individual components of complex multi-NV spectra

## Working with Parameter Constraints

When using models with fitting routines, you can specify constraints:

In [ ]:
# Example constraint dictionary for ESR14N
constraints = {
    'center': [2.8e9, 2.9e9],      # Center frequency bounds (Hz)
    'width': [1e6, 1e7],           # Linewidth bounds (Hz)
    'contrast': [0.0, 1.0],        # Contrast bounds (0-1)
    'offset': [-0.1, 0.1],         # Offset bounds (0-1)
}

# Convert to constraint array for fitting
model = ModelRegistry.get('ESR14N')
constraint_array = model.get_constraint_array(constraints)

print("ESR14N Parameter Constraints:")
print(f"Model has {model.n_parameters} parameters")
print(f"Constraint array length: {len(constraint_array)} (2 values per parameter)")
print(f"Parameters: {model.parameters_unique}")
print(f"Constraint array: {constraint_array}")

# Show how constraints are organized
print("\nConstraint organization (min, max pairs):")
for i, param in enumerate(model.parameters_unique):
    min_val = constraint_array[2*i]
    max_val = constraint_array[2*i + 1]
    print(f"  {param}: [{min_val:.1e}, {max_val:.1e}]")

## Mathematical Formulation

All models implement Lorentzian absorption lines. Here's the mathematical basis:

In [ ]:
# Demonstrate the mathematical formulation
print("Mathematical Formulation of ODMR Models:")
print("\nSingle Lorentzian (ESRSINGLE):")
print("f(x) = 1 + offset - (contrast × width² / ((x - center)² + width²))")

print("\nMulti-peak models (ESR14N, ESR15N):")
print("f(x) = 1 + offset - Σᵢ (contrastᵢ × width² / ((x - posᵢ)² + width²))")
print("where posᵢ are the hyperfine-shifted resonance positions")

print("\nHyperfine positions:")
print("ESR14N: center-Ahyp, center, center+Ahyp (mI = -1, 0, +1)")
print("ESR15N: center-Ahyp, center+Ahyp (mI = -1/2, +1/2)")

print(f"\nHyperfine constants:")
print(f"14N: {all_models['ESR14N']['hyp']} Hz")
print(f"15N: {all_models['ESR15N']['hyp']} Hz")

# Show parameter order for each model
print("\nParameter order for each model:")
for model_name in ['ESRSINGLE', 'ESR15N', 'ESR14N']:
    model = ModelRegistry.get(model_name)
    print(f"{model_name}: {model.parameters_unique}")

## Integration with QDMpy Workflow

These models integrate seamlessly with QDMpy's fitting and measurement infrastructure:

```python
from QDMpy import Measurement
from QDMpy.models import ModelRegistry

# In a typical workflow:
# 1. Load ODMR data
# 2. Select appropriate model
model = ModelRegistry.get('ESR14N')  # or 'ESR15N', 'ESRSINGLE'

# 3. The model is automatically used by fitting routines
# measurement.fit_odmr(model=model)
```

## Summary

QDMpy's model system provides:
- **Three robust models** covering common NV center configurations
- **Physics-based implementations** with proper hyperfine splitting  
- **Flexible parameter management** with constraint support
- **GPU-optimized performance** for large-scale fitting

**Note**: The models are pre-implemented and optimized for the pyGpufit backend. Custom model creation requires modifications to both QDMpy and pyGpufit, so it's recommended to use the existing models which cover the most common ODMR scenarios.

Choose the model that best matches your experimental system and data quality!